# Data Engineering Project – Notebook Notes

## 📌 Notebook Overview

**Purpose:**
Briefly describe what this notebook does and where it fits in the overall data engineering pipeline.

**Pipeline Stage:**
`[Extraction / Transformation / Loading / Profiling / Analysis / Other]`

**Input:**

* Source:
* Dataset:
* Format:

**Output:**

* Destination:
* Format:
* Tables / files created:

---


# 1. Environment Setup

### Objective

Set up the Python environment and load the required GSS project id, bucket id etc. This is applicable only for this note book. Environment for docker is handled seperately.
---

In [ ]:
#Install the kagglehub package with pip:
%pip install kagglehub

In [3]:
# Load environment (only for notebook not for docker)
import os
from dotenv import load_dotenv

load_dotenv("../.env")


GCP_PROJECT_ID = os.getenv("GCP_PROJECT_ID")
GCS_BUCKET_NAME = os.getenv("GCS_BUCKET_NAME")
GCS_DATASET_ID = os.getenv("GCS_DATASET_ID")


print(GCP_PROJECT_ID)
print(GCS_BUCKET_NAME)
print(GCS_DATASET_ID)

my-project-bigdata-505312
olistcontainer
olist_raw


# 2. Data Extraction

### Objective

The below code section writes the content of the cell to "kaggle_extract.py" located in "../src/extract/kaggle_extract.py" this note book serves as one point contact so that we no need to open the source file and edit. The below script downloads data from kaggle (9 csv files) cache it locally and then loads them to GCS Bucket. Since dagster is used for scheduling "extract_kaggle_to_gcs" this function inside this script is called in dagster pipeline. We run the below code cell only when we make any code change written to the file. 

In [19]:
%%writefile ../src/extract/kaggle_extract.py
# you could use kagglehub to download the dataset and then the Google Cloud Storage Python library to upload it.
import os

import kagglehub
from google.cloud import storage

print("kaggle-setup")
def extract_kaggle_to_gcs():

    GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]
    GCS_BUCKET_NAME = os.environ["GCS_BUCKET_NAME"]

    print("GCP Project:", GCP_PROJECT_ID)
    print("GCS Bucket:", GCS_BUCKET_NAME)

    # 1. Download dataset from Kaggle
    path = kagglehub.dataset_download(
        "olistbr/brazilian-ecommerce"
    )

    print("Downloaded to:", path)

    # 2. Connect to GCS
    client = storage.Client(project=GCP_PROJECT_ID)

    bucket = client.bucket(GCS_BUCKET_NAME)

    print("Connected to bucket:", bucket.name)

    # 3. Upload files
    for filename in os.listdir(path):

        local_file = os.path.join(path, filename)

        if os.path.isfile(local_file):

            blob = bucket.blob(
                f"raw/kaggle/{filename}"
            )

            blob.upload_from_filename(local_file)

            print(f"Uploaded: {filename}")

Overwriting ../src/extract/kaggle_extract.py


# 2. Data loading

### Objective

def load_gcs_to_bigquery():
The below code section writes the content of the cell to "kaggle_load.py" located in "../src/extract/kaggle_load.py" this note book serves as one point contact so that we no need to open the source file and edit. The below script loads CSV raw data from GCS and load it as tables to big querry. Since dagster is used for scheduling "extract_kaggle_to_gcs" this function inside this script is called in dagster pipeline. We run the below code cell only when we make any code change written to the file. 

In [2]:
%%writefile ../src/loader/kaggle_load.py
# Create big querry data set 
import os

from google.cloud import bigquery


def load_gcs_to_bigquery():

    GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]
    GCS_BUCKET_NAME = os.environ["GCS_BUCKET_NAME"]
    GCS_DATASET_ID = os.environ["GCS_DATASET_ID"]

    print("GCP Project:", GCP_PROJECT_ID)
    print("GCS Bucket:", GCS_BUCKET_NAME)
    print("BigQuery Dataset:", GCS_DATASET_ID)

    client = bigquery.Client(project=GCP_PROJECT_ID)

    print("Connected to BigQuery!")

    # Create BigQuery dataset
    dataset_ref = f"{GCP_PROJECT_ID}.{GCS_DATASET_ID}"

    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = "asia-southeast1"

    client.create_dataset(dataset, exists_ok=True)

    print(f"BigQuery dataset ready: {dataset_ref}")

    files = [
        "olist_sellers_dataset.csv",
        "product_category_name_translation.csv",
        "olist_orders_dataset.csv",
        "olist_order_items_dataset.csv",
        "olist_customers_dataset.csv",
        "olist_geolocation_dataset.csv",
        "olist_order_payments_dataset.csv",
        "olist_order_reviews_dataset.csv",
        "olist_products_dataset.csv"
    ]

    for filename in files:

        table_name = filename.replace(".csv", "")
        table_name = table_name.replace("_dataset", "")

        table_id = (
            f"{GCP_PROJECT_ID}."
            f"{GCS_DATASET_ID}."
            f"{table_name}"
        )

        uri = (
            f"gs://{GCS_BUCKET_NAME}/"
            f"raw/kaggle/{filename}"
        )

        job_config = bigquery.LoadJobConfig(
            source_format=bigquery.SourceFormat.CSV,
            skip_leading_rows=1,
            autodetect=True,
            allow_quoted_newlines=True,
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
        )

        print(f"Loading {filename} → {table_id}")

        load_job = client.load_table_from_uri(
            uri,
            table_id,
            job_config=job_config
        )

        load_job.result()

        print(f"✓ Loaded: {table_name}")

Overwriting ../src/loader/kaggle_load.py


# 3. Configure Raw BigQuery Tables as dbt Sources

The raw Olist data is already loaded into the BigQuery `olist_raw` dataset.

The next step is to configure these raw tables as **dbt sources** so that dbt can reference them in staging models.

### 1. Check the Raw Tables in BigQuery

Run the following command inside the Docker `app` container to list the tables available in the `olist_raw` dataset:

```bash
python -c "from google.cloud import bigquery; c=bigquery.Client(); tables=c.list_tables('my-project-bigdata-505312.olist_raw'); print('\n'.join(t.table_id for t in tables))"
```

Expected output:

```text
olist_customers
olist_geolocation
olist_order_items
olist_order_payments
olist_order_reviews
olist_orders
olist_products
olist_sellers
product_category_name_translation
```

This confirms that the raw Olist tables are available in BigQuery.

### 2. Create the dbt Staging Directory

Create the staging directory inside the dbt project:

```bash
mkdir -p models/staging
```

### 3. Create the `sources.yml` File

Create a dbt source configuration file:

```bash
cat > models/staging/sources.yml <<'EOF'
version: 2

sources:
  - name: olist
    database: my-project-bigdata-505312
    schema: olist_raw
    tables:
      - name: olist_customers
      - name: olist_geolocation
      - name: olist_order_items
      - name: olist_order_payments
      - name: olist_order_reviews
      - name: olist_orders
      - name: olist_products
      - name: olist_sellers
      - name: product_category_name_translation
EOF
```

### 4. Understanding the Source Configuration

The `sources.yml` file tells dbt where the raw data is located.

```yaml
sources:
  - name: olist
```

`olist` is the **source name** used inside dbt.

For example:

```sql
{{ source('olist', 'olist_orders') }}
```

The BigQuery location is defined by:

```yaml
database: my-project-bigdata-505312
schema: olist_raw
```

Here:

* `database` → BigQuery project ID
* `schema` → BigQuery dataset name
* `tables` → Raw tables available in the dataset

The configuration therefore maps:

```text
dbt source: olist
        │
        ▼
BigQuery project: my-project-bigdata-505312
        │
        ▼
BigQuery dataset: olist_raw
        │
        ├── olist_customers
        ├── olist_geolocation
        ├── olist_order_items
        ├── olist_order_payments
        ├── olist_order_reviews
        ├── olist_orders
        ├── olist_products
        ├── olist_sellers
        └── product_category_name_translation
```

These source definitions can then be referenced by the staging models using:

```sql
{{ source('olist', 'table_name') }}
```

For example:

```sql
FROM {{ source('olist', 'olist_orders') }}
```



# 4. Data Profiling

A profiling script is included to validate the loaded data.

Example:

```bash
docker compose exec app python /module-2-project/src/profiling/profile_orders.py
```

The profiling checks include:

* Row count
* Order status distribution
* Missing values
* Data consistency
* Other basic data quality characteristics

Example profiling result:

```text
Total rows: 99,441

delivered: 96,478
shipped: 1,107
canceled: 625
unavailable: 609
```

The profiling step helps verify that the ingestion process completed successfully before transformation.

---


In [34]:
%%writefile ../src/profiling/profile_orders.py 

from google.cloud import bigquery
import os


def profile_orders():

    GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]
    GCS_DATASET_ID = os.environ["GCS_DATASET_ID"]

    # ---------------------------------------------------------
    # Configuration
    # ---------------------------------------------------------

    TABLE = "olist_orders"

    TABLE_ID = f"{GCP_PROJECT_ID}.{GCS_DATASET_ID}.{TABLE}"

    # ---------------------------------------------------------
    # BigQuery connection
    # ---------------------------------------------------------

    client = bigquery.Client(project=GCP_PROJECT_ID)

    print("=" * 70)
    print("DATA PROFILING REPORT")
    print("=" * 70)
    print(f"Table: {TABLE_ID}")
    print()

    # ---------------------------------------------------------
    # 1. Row count
    # ---------------------------------------------------------

    print("1. ROW COUNT")
    print("-" * 70)

    query = f"""
    SELECT COUNT(*) AS row_count
    FROM `{TABLE_ID}`
    """

    result = list(client.query(query).result())
    row_count = result[0]["row_count"]

    print(f"Total rows: {row_count:,}")
    print()

    # ---------------------------------------------------------
    # 2. Order status distribution
    # ---------------------------------------------------------

    print("2. ORDER STATUS DISTRIBUTION")
    print("-" * 70)

    query = f"""
    SELECT
        order_status,
        COUNT(*) AS count
    FROM `{TABLE_ID}`
    GROUP BY order_status
    ORDER BY count DESC
    """

    results = client.query(query).result()

    for row in results:
        print(f"{row['order_status']}: {row['count']:,}")

    print()

    # ---------------------------------------------------------
    # 3. NULL value analysis
    # ---------------------------------------------------------

    print("3. NULL VALUE ANALYSIS")
    print("-" * 70)

    query = f"""
    SELECT
        COUNTIF(order_id IS NULL) AS order_id_nulls,
        COUNTIF(customer_id IS NULL) AS customer_id_nulls,
        COUNTIF(order_status IS NULL) AS status_nulls,
        COUNTIF(order_purchase_timestamp IS NULL) AS purchase_nulls,
        COUNTIF(order_approved_at IS NULL) AS approved_nulls,
        COUNTIF(order_delivered_carrier_date IS NULL) AS carrier_nulls,
        COUNTIF(order_delivered_customer_date IS NULL) AS delivered_nulls,
        COUNTIF(order_estimated_delivery_date IS NULL) AS estimated_nulls
    FROM `{TABLE_ID}`
    """

    row = list(client.query(query).result())[0]

    null_columns = {
        "order_id": row["order_id_nulls"],
        "customer_id": row["customer_id_nulls"],
        "order_status": row["status_nulls"],
        "order_purchase_timestamp": row["purchase_nulls"],
        "order_approved_at": row["approved_nulls"],
        "order_delivered_carrier_date": row["carrier_nulls"],
        "order_delivered_customer_date": row["delivered_nulls"],
        "order_estimated_delivery_date": row["estimated_nulls"],
    }

    for column, count in null_columns.items():
        percentage = (count / row_count * 100) if row_count else 0
        print(f"{column:40} {count:8,} ({percentage:6.2f}%)")

    print()

    # ---------------------------------------------------------
    # 4. Duplicate order IDs
    # ---------------------------------------------------------

    print("4. DUPLICATE ORDER IDs")
    print("-" * 70)

    query = f"""
    SELECT
        order_id,
        COUNT(*) AS count
    FROM `{TABLE_ID}`
    GROUP BY order_id
    HAVING COUNT(*) > 1
    ORDER BY count DESC
    LIMIT 20
    """

    results = list(client.query(query).result())

    if not results:
        print("No duplicate order_ids found.")
    else:
        print("Duplicate order_ids detected.")
        print("Showing up to 20 examples:")
        print()

        for row in results:
            print(
                f"order_id: {row['order_id']} | "
                f"count: {row['count']}"
            )

    print()

    # ---------------------------------------------------------
    # 5. Purchase date range
    # ---------------------------------------------------------

    print("5. PURCHASE DATE RANGE")
    print("-" * 70)

    query = f"""
    SELECT
        MIN(order_purchase_timestamp) AS earliest_order,
        MAX(order_purchase_timestamp) AS latest_order
    FROM `{TABLE_ID}`
    """

    row = list(client.query(query).result())[0]

    print(f"Earliest order: {row['earliest_order']}")
    print(f"Latest order:   {row['latest_order']}")
    print()

    # ---------------------------------------------------------
    # 6. Invalid delivery dates
    # ---------------------------------------------------------

    print("6. INVALID DELIVERY DATES")
    print("-" * 70)

    query = f"""
    SELECT COUNT(*) AS invalid_delivery_dates
    FROM `{TABLE_ID}`
    WHERE order_delivered_customer_date < order_purchase_timestamp
    """

    row = list(client.query(query).result())[0]

    print(
        f"Orders delivered before purchase: "
        f"{row['invalid_delivery_dates']:,}"
    )
    print()

    # ---------------------------------------------------------
    # 7. Invalid estimated delivery dates
    # ---------------------------------------------------------

    print("7. INVALID ESTIMATED DELIVERY DATES")
    print("-" * 70)

    query = f"""
    SELECT COUNT(*) AS invalid_count
    FROM `{TABLE_ID}`
    WHERE DATE(order_estimated_delivery_date)
          < DATE(order_purchase_timestamp)
    """

    row = list(client.query(query).result())[0]

    print(
        f"Orders with estimated delivery before purchase: "
        f"{row['invalid_count']}"
    )

    # ---------------------------------------------------------
    # Profiling complete
    # ---------------------------------------------------------

    print("=" * 70)
    print("PROFILING COMPLETE")
    print("=" * 70)

Overwriting ../src/profiling/profile_orders.py




# 5. dbt Setup

dbt is used for transforming the raw BigQuery data into analytical models.

The dbt project is separate from the extraction and orchestration code but operates on the BigQuery data produced by the ingestion pipeline.

---

## 5.1 Install dbt

The Docker environment contains:

```text
dbt-core
dbt-bigquery
```

The project was configured with versions similar to:

```text
dbt-core: 1.12.3
dbt-bigquery: 1.12.0
```

Check the installation:

```bash
docker compose exec app dbt --version
```

---

# 6. Create the dbt Project

Inside the container:

```bash
docker compose exec app bash
```

Navigate to the dbt directory:

```bash
cd /module-2-project/dbt
```

A dbt project contains files such as:

```text
dbt/
├── dbt_project.yml
├── models/
│   ├── staging/
│   └── marts/
├── tests/
├── macros/
└── seeds/
```

---

# 7. dbt BigQuery Profile

dbt needs a profile to connect to BigQuery.

The profile specifies:

* GCP project
* BigQuery dataset
* Authentication method
* Location
* Target environment

Example structure:

```yaml
olist_project:
  target: dev

  outputs:
    dev:
      type: bigquery
      method: service-account
      project: my-project-bigdata-505312
      dataset: olist_raw
      keyfile: /workspace/gcp-key.json
      location: asia-southeast1
      threads: 4
```

The exact dataset and location should match the BigQuery configuration.

---

# 8. Validate dbt Connection

Run:

```bash
docker compose exec app dbt debug
```

A successful configuration should report that the dbt project and BigQuery connection are valid.

---

# 9. Run dbt Models

To execute the transformation models:

```bash
docker compose exec app dbt run
```

dbt reads the raw BigQuery data and creates the configured transformation models.

---

# 10. Run dbt Tests

Data quality tests can be executed using:

```bash
docker compose exec app dbt test
```

Typical dbt tests include:

* `not_null`
* `unique`
* `accepted_values`
* Referential integrity relationships

Example:

```yaml
columns:
  - name: order_id
    tests:
      - not_null
      - unique
```

---

# 11. Run dbt Build

When both models and tests need to be executed:

```bash
docker compose exec app dbt build
```

`dbt build` provides a convenient way to execute the configured dbt resources and associated tests.

---

## Ask dbt to Parse the Project

Before running the dbt models, first ask dbt to parse and validate the project.

Run:

```bash
dbt parse
```

This is an important concept.

`dbt parse` is a safe first validation step. It checks whether dbt can read and understand the project configuration, models, sources, and dependencies.

The overall flow is:

```text
Raw data
   ↓
dbt parse          ← "Can dbt understand my project?"
   ↓
dbt source         ← "Where is my raw data?"
   ↓
Staging models     ← "Clean and standardize the data"
   ↓
Analytics models   ← "Transform it for business use"
```

### Go to the Docker Environment

From the host machine:

```bash
docker compose exec app bash
```

You should now be inside the Docker container.

### Navigate to the dbt Project

Navigate to the directory containing `dbt_project.yml`:

```bash
cd dbt/olist_dbt/
```

Run:

```bash
dbt parse
```

> **Rule to remember:**
> Run dbt commands from the directory containing `dbt_project.yml`.

### Verify dbt Sources

After parsing the project, verify that dbt recognizes the configured sources:

```bash
dbt ls --resource-type source
```

If the sources are recognized, dbt will list the configured source definitions.

This confirms that dbt can identify the raw data sources before the staging models are built.


## Next Step: Creating staging Models

After verifying that dbt can recognize the raw sources, the next step is to create the staging models.

### 1. Inspect the Raw Table

Before creating the model, inspect the raw table to understand:

* Available columns
* Data types
* Potential cleaning requirements
* Fields needed for the staging model

Run the following command inside the Docker container:

```bash
python -c "from google.cloud import bigquery; c=bigquery.Client(); t=c.get_table('my-project-bigdata-505312.olist_raw.olist_orders'); print('\n'.join(f'{x.name}: {x.field_type}' for x in t.schema))"
```

### 2. Create `stg_orders.sql`

Create the staging model:

```bash
cat > models/staging/stg_orders.sql <<'EOF'

SELECT

    order_id,

    customer_id,

    order_status,

    order_purchase_timestamp,

    order_approved_at,

    order_delivered_carrier_date,

    order_delivered_customer_date,

    order_estimated_delivery_date

FROM {{ source('olist', 'olist_orders') }}

EOF
```

> The `stg_orders.sql` file is also available in the notebook/project files and can be edited from the notebook if preferred.

### 3. Verify the Model

Check that the file was created correctly:

```bash
cat models/staging/stg_orders.sql
```

### 4. Compile the dbt Model

`dbt compile` takes the dbt SQL, resolves dbt-specific syntax such as `{{ source(...) }}`, and generates the actual SQL that BigQuery understands.

The flow is:

```text
Your dbt SQL
     ↓
dbt compile
     ↓
Actual BigQuery SQL
```

Run:

```bash
dbt compile --select stg_orders
```

### 5. Run the Model

`dbt run` asks dbt to execute the model and create the transformed object in BigQuery.

Run:

```bash
dbt run --select stg_orders
```

The resulting model will be created in the configured target dataset.

### 6. Verify the Result in BigQuery

Query the newly created `stg_orders` model to confirm that the data looks correct:

```bash
python -c "from google.cloud import bigquery; c=bigquery.Client(); q='SELECT * FROM `my-project-bigdata-505312.olist_analytics.stg_orders` LIMIT 5'; rows=c.query(q).result(); [print(dict(r)) for r in rows]"
```

### 7. Profile the Staging Model

After verifying the sample data, run the four profiling checks discussed earlier:

1. **Row count**
2. **Order-status distribution**
3. **NULL counts**
4. **Duplicate `order_id`**

These checks help confirm the quality and consistency of the staging model before building the downstream analytical models.


In [24]:
%%writefile ../dbt/Olist_dbt/models/staging/stg_orders.sql 

SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
FROM {{ source('olist', 'olist_orders') }}

Overwriting ../dbt/Olist_dbt/models/staging/stg_orders.sql



## run staging querry

```bash
docker compose exec app dbt run --select stg_orders
```

## Building the Analytical Model

The `monthly_order_volume` model is the analytical model in this project.

The data flow is:

```text
olist_raw
    │
    ▼
stg_orders                  ← VIEW
    │
    ▼
monthly_order_volume        ← TABLE
    │
    ▼
Streamlit
```

### 1. Create the `marts` Directory

The analytical models are stored separately from the staging models.

From the dbt project directory:

```bash
mkdir -p models/marts
```

The project structure will look like:

```text
models/
├── staging/
│   └── stg_orders.sql
│
└── marts/
    └── monthly_order_volume.sql
```

### 2. Create the Monthly Order Volume Model

Create the SQL file:

```bash
cat > models/marts/monthly_order_volume.sql <<'EOF'

SELECT
    DATE_TRUNC(DATE(order_purchase_timestamp), MONTH) AS order_month,
    COUNT(*) AS order_volume
FROM {{ ref('stg_orders') }}
GROUP BY order_month
ORDER BY order_month

EOF
```

The important concept here is the use of:

```sql
{{ ref('stg_orders') }}
```

`ref()` tells dbt that `monthly_order_volume` depends on the `stg_orders` model.

dbt can therefore understand the dependency:

```text
stg_orders
     ↓
monthly_order_volume
```

### 3. Configure the Mart Table

Configure the analytical model as a **table** in `dbt_project.yml`.

Under the `models` configuration, add the `marts` configuration:

```yaml
models:
  olist_dbt:
    staging:
      +materialized: view

    marts:
      +materialized: table
```

This means:

* Models under `staging/` are materialized as **views**
* Models under `marts/` are materialized as **tables**

The resulting BigQuery objects are:

```text
olist_raw
    │
    ▼
stg_orders
    VIEW
    │
    ▼
monthly_order_volume
    TABLE
```

### 4. Verify the Model

Check the SQL file:

```bash
cat models/marts/monthly_order_volume.sql
```

Then check that dbt recognizes the model:

```bash
dbt ls --select monthly_order_volume
```

### 5. Compile the Analytical Model

Before running it, compile the model:

```bash
dbt compile --select monthly_order_volume
```

This allows dbt to resolve the `ref('stg_orders')` dependency and generate the SQL that will be executed in BigQuery.

### 6. Run the Analytical Model

Run:

```bash
dbt run --select monthly_order_volume
```

dbt will build the `monthly_order_volume` table in the configured analytical dataset.

The final pipeline is:

```text
Raw Olist Data
      │
      ▼
olist_raw.olist_orders
      │
      ▼
stg_orders
      │
      │  VIEW
      ▼
monthly_order_volume
      │
      │  TABLE
      ▼
Streamlit Dashboard
```



### monthly_order_volume.sql
below is the section where you can modify the .sql file

In [23]:
%%writefile ../dbt/Olist_dbt/models/marts/monthly_order_volume.sql

{{ config(materialized='table') }}

SELECT
    DATE_TRUNC(
        DATE(order_purchase_timestamp),
        MONTH
    ) AS order_month,

    COUNT(DISTINCT order_id) AS order_count

FROM {{ ref('stg_orders') }}

GROUP BY order_month
ORDER BY order_month

Overwriting ../dbt/Olist_dbt/models/marts/monthly_order_volume.sql


## dbt run

```bash
docker compose exec app dbt run --select monthly_order_volume
```

BigQuery
└── olist_analytics
    └── monthly_order_volume

##  look at actual analytical data 
##  preview 5 
```bash
docker compose exec app dbt show --select monthly_order_volume
```
## view full table 
```bash
docker compose exec app dbt show --select monthly_order_volume --limit 30
```

## Streamlit
- Create streamlit folder 
- Create app.py 
- Modify compose.yml to add streamlit
- rebuild docker 
```bash
docker compose up -d --build
```
- run streamlit in docker env
```bash
docker compose exec app ls -la /module-2-project/streamlit
```

Step 1  → Streamlit runs
            ↓
Step 2  → Connect to BigQuery
            ↓
Step 3  → Read monthly_order_volume
            ↓
Step 4  → Display the 25 months
            ↓
Step 5  → Add the chart
            ↓
Step 6  → Add business-event analysis

- add streamlit in docker requirements.txt
- rebuild docker 
```bash
docker compose up -d --build
```
- run streamlit
```bash
docker compose exec app streamlit run /module-2-project/streamlit/app.py --server.address=0.0.0.0
```
- Since docker is used streamlit in docker port need to be mapped to mac port before we can see 
- add ports in compose.yml

## below is the app.py section in this notebook

In [21]:
%%writefile ../streamlit/app.py

import os

import pandas as pd
import streamlit as st
from google.cloud import bigquery


# --------------------------------------------------
# Streamlit page configuration
# --------------------------------------------------

st.set_page_config(
    page_title="Olist Analytics",
    page_icon="📊",
    layout="wide"
)

st.title("📊 Olist Analytics Dashboard")
st.write("Historical monthly order volume")


# --------------------------------------------------
# BigQuery configuration
# --------------------------------------------------

PROJECT_ID = os.environ["GCP_PROJECT_ID"]

TABLE_ID = (
    f"{PROJECT_ID}."
    "olist_analytics."
    "monthly_order_volume"
)


# --------------------------------------------------
# Query BigQuery
# --------------------------------------------------

@st.cache_data
def load_data():

    client = bigquery.Client(project=PROJECT_ID)

    query = f"""
        SELECT
            order_month,
            order_count
        FROM `{TABLE_ID}`
        ORDER BY order_month
    """

    df = client.query(query).to_dataframe()

    return df


# --------------------------------------------------
# Load data
# --------------------------------------------------

df = load_data()


# --------------------------------------------------
# Display data
# --------------------------------------------------

st.subheader("Monthly Order Volume")

st.dataframe(
    df,
    use_container_width=True
)


# --------------------------------------------------
# Display chart
# --------------------------------------------------

st.subheader("Order Volume Trend")

chart_df = df.set_index("order_month")

st.line_chart(
    chart_df["order_count"]
)


# --------------------------------------------------
# Data interpretation
# --------------------------------------------------

st.info(
    "Note: September and October 2018 contain incomplete "
    "data because the source dataset ends in October 2018."
)



Overwriting ../streamlit/app.py


## Next Step: Introduce Dagster for Orchestration

The next step is to introduce **Dagster** as the orchestration layer for the data engineering pipeline.

Dagster will be responsible for controlling the execution of the different pipeline components, managing dependencies, scheduling jobs, handling retries, monitoring pipeline runs, and providing data lineage.

### Target Architecture

```text
                         ┌─────────────────────┐
                         │       DAGSTER       │
                         │                     │
                         │  Controls order     │
                         │  Dependencies       │
                         │  Retries            │
                         │  Scheduling         │
                         │  Monitoring         │
                         │  Data lineage       │
                         └─────────┬───────────┘
                                   │
                  ┌────────────────┼────────────────┐
                  ▼                ▼                ▼
               Extract          Profile           dbt
                  │                │                │
                  └────────────────┼────────────────┘
                                   │
                                   ▼
                              Analytics
                                   │
                                   ▼
                              Streamlit
```

The objective is to move from manually executing individual pipeline components to having Dagster manage the overall workflow.

---

## 1. Create the Dagster Framework

The project is organised into separate components for extraction, loading, profiling, transformation, visualization, and orchestration.

```text
Module-2-Project/
│
├── src/
│   ├── extract/
│   ├── load/
│   ├── profiling/
│   └── ...
│
├── dbt/
│   └── olist_dbt/
│
├── streamlit/
│   └── app.py
│
└── dagster/
    ├── definitions.py
    └── ...
```

The `dagster/` directory contains the Dagster-specific orchestration code.

---

## 2. Check the Existing Dagster Installation

Before adding Dagster to the project, verify whether it is already installed in the Docker environment.

Run:

```bash
dagster --version
```

If Dagster is not installed, the output will be similar to:

```text
root@a3af08d8685c:/module-2-project# dagster --version

bash: dagster: command not found
```

This indicates that Dagster needs to be added to the project dependencies.

---

## 3. Add Dagster to `requirements.txt`

Add the following packages to `requirements.txt`:

```text
dagster
dagster-webserver
```

`dagster` provides the core orchestration framework, while `dagster-webserver` provides the Dagster web UI.

---

## 4. Rebuild the Docker Image

Because the Python dependencies have changed, rebuild the Docker image.

Stop the existing containers:

```bash
docker compose down
```

Then rebuild and start the application:

```bash
docker compose up -d --build
```

The `--build` option ensures that the new Dagster dependencies are installed into the Docker image.

---

## 5. Verify Dagster Installation

After the container has started, verify that Dagster is available inside the container:

```bash
docker compose exec app dagster --version
```

Also verify that the Python package can be imported:

```bash
docker compose exec app python -c "import dagster; print(dagster.__version__)"
```

Both commands should return a Dagster version number.

---

## 6. Add the Dagster Directory to Docker Compose

The local Dagster directory needs to be available inside the Docker container.

Add the following volume mapping to `docker-compose.yml`:

```yaml
volumes:
  - ./dagster:/module-2-project/dagster
```

This maps:

```text
Local machine
./dagster
      │
      │ Docker volume
      ▼
Container
/module-2-project/dagster
```

This allows Dagster code developed in the repository to be accessed from inside the container.

---

## 7. Create the Dagster Directory

If the directory does not already exist, create it from the project root:

```bash
mkdir -p dagster
```

The project structure now becomes:

```text
Module-2-Project/
│
├── src/
├── dbt/
├── streamlit/
│
└── dagster/
```

---

## 8. Restart the Docker Environment

Restart the Docker environment so that the new volume mapping is applied:

```bash
docker compose down
docker compose up -d
```

---

## 9. Verify the Dagster Directory Inside the Container

Confirm that the local Dagster directory is visible inside the container:

```bash
docker compose exec app ls -la /module-2-project/dagster
```

The directory should be accessible from:

```text
/module-2-project/dagster
```

---

## 10. Create `definitions.py`

Create the main Dagster definitions file:

```text
dagster/
└── definitions.py
```

`definitions.py` is the entry point where the project's Dagster assets, jobs, schedules, resources, and other orchestration definitions can be registered.

The initial Dagster implementation can then be extended to integrate the existing pipeline components:

```text
Python Extraction
        │
        ▼
      Load
        │
        ▼
    Profiling
        │
        ▼
      dbt
        │
        ▼
   Analytics
        │
        ▼
   Streamlit
```

Dagster will progressively take responsibility for orchestrating these stages rather than requiring them to be executed manually.

---

## 11. Result

After introducing Dagster, the architecture becomes:

```text
                    ┌──────────────────────┐
                    │       DAGSTER        │
                    │                      │
                    │  Orchestration       │
                    │  Dependencies        │
                    │  Scheduling          │
                    │  Retries             │
                    │  Monitoring          │
                    │  Data lineage        │
                    └──────────┬───────────┘
                               │
             ┌─────────────────┼─────────────────┐
             │                 │                 │
             ▼                 ▼                 ▼
         Extraction         Profiling           dbt
             │                 │                 │
             └─────────────────┼─────────────────┘
                               │
                               ▼
                          Analytics
                               │
                               ▼
                          Streamlit
```

The project has now established the **Dagster orchestration framework**. The next step is to define the actual Dagster assets and dependencies so that the existing extraction, loading, profiling, and dbt components can be executed as a coordinated pipeline.


In [44]:
%%writefile ../dagster/definitions.py
# Create Dagster assets in this file

from pathlib import Path

from dagster import asset, Definitions, ScheduleDefinition, define_asset_job, AssetSelection
from dagster_dbt import DbtCliResource, DagsterDbtTranslator, dbt_assets

from src.extract.kaggle_extract import extract_kaggle_to_gcs
from src.loader.kaggle_load import load_gcs_to_bigquery
from src.profiling.profile_orders import profile_orders


# ---------------------------------------------------------
# Existing pipeline assets
# ---------------------------------------------------------

@asset
def kaggle_to_gcs():
    extract_kaggle_to_gcs()


@asset(deps=[kaggle_to_gcs])
def gcs_to_bigquery_raw():
    load_gcs_to_bigquery()


@asset(deps=[gcs_to_bigquery_raw])
def profile_orders_asset():
    profile_orders()


# ---------------------------------------------------------
# DBT configuration
# ---------------------------------------------------------

dbt_project_dir = Path("/module-2-project/dbt/olist_dbt")

dbt = DbtCliResource(
    project_dir=dbt_project_dir,
    profiles_dir="/module-2-project/dbt/profiles",
)


# ---------------------------------------------------------
# DBT assets
# ---------------------------------------------------------
class CustomDbtTranslator(DagsterDbtTranslator):

    def get_asset_spec(self, manifest, unique_id, project):

        spec = super().get_asset_spec(
            manifest,
            unique_id,
            project
        )

        if spec.key.to_user_string() == "stg_orders":
            spec = spec.merge_attributes(
                deps=[gcs_to_bigquery_raw]
            )

        return spec

@dbt_assets(
    manifest=dbt_project_dir / "target" / "manifest.json",
    dagster_dbt_translator=CustomDbtTranslator(),
)
def olist_dbt_assets(context, dbt: DbtCliResource):
    yield from dbt.cli(["build"], context=context).stream()

# ---------------------------------------------------------
# Dagster job and schedule
# ---------------------------------------------------------

daily_pipeline_job = define_asset_job(
    name="daily_olist_pipeline",
    selection=AssetSelection.all(),
)

daily_olist_schedule = ScheduleDefinition(
    job=daily_pipeline_job,
    cron_schedule="0 2 * * *",
    execution_timezone="Asia/Singapore",
)

# ---------------------------------------------------------
# Dagster definitions
# ---------------------------------------------------------

defs = Definitions(
    assets=[
        kaggle_to_gcs,
        gcs_to_bigquery_raw,
        profile_orders_asset,
        olist_dbt_assets,
    ],
    resources={
        "dbt": dbt,
    },
    jobs=[
        daily_pipeline_job,
    ],
    schedules=[
        daily_olist_schedule,
    ],
)

Overwriting ../dagster/definitions.py


# Running and Extending Dagster in the Docker Environment

Once the Dagster framework was created, the next step was to progressively integrate the existing data engineering pipeline into Dagster.

The integration was performed incrementally:

```text
Kaggle
   │
   ▼
kaggle_to_gcs
   │
   ▼
GCS
   │
   ▼
gcs_to_bigquery_raw
   │
   ▼
BigQuery RAW
   │
   ├──────────────► Profiling
   │
   ▼
  dbt
   │
   ▼
stg_orders
   │
   ▼
monthly_order_volume
   │
   ▼
Streamlit
```

The pipeline was built and tested one dependency at a time.

---

# 1. Run Dagster Definitions in Docker

The Dagster code is executed inside the Docker container.

Enter the container:

```bash
docker compose exec app bash
```

Navigate to the Dagster directory:

```bash
cd /module-2-project/dagster
```

Run the definitions file directly to verify that it can be imported:

```bash
python definitions.py
```

Expected output:

```text
definitions
```

This confirms that the Python file can be executed successfully.

---

# 2. Verify Dagster Assets

Before integrating the complete pipeline, first verify that Dagster can discover the assets defined in `definitions.py`.

List the assets:

```bash
dagster asset list -f definitions.py -d /module-2-project
```

Materialize the assets:

```bash
dagster asset materialize -f definitions.py -d /module-2-project
```

This provides an initial validation that Dagster is able to load and execute the asset definitions.

---

# 3. Add the Kaggle → GCS Pipeline

The first actual pipeline operation integrated into Dagster was the existing **Kaggle → GCS** process.

The existing Python extraction/upload functionality was tested independently first and then added as a Dagster asset.

The first asset was:

```text
kaggle_to_gcs
```

The resulting flow was:

```text
Kaggle
   │
   ▼
kaggle_to_gcs
   │
   ▼
GCS
```

### Verify the asset

```bash
docker compose exec app bash -c \
"cd /module-2-project/dagster && \
dagster asset list -f definitions.py -d /module-2-project"
```

Expected asset:

```text
kaggle_to_gcs
```

### Materialize the asset

```bash
docker compose exec app bash -c \
"cd /module-2-project/dagster && \
dagster asset materialize -f definitions.py -d /module-2-project --select kaggle_to_gcs"
```

This verifies that Dagster can execute the Kaggle-to-GCS operation.

---

# 4. Add GCS → BigQuery RAW Dependency

The next stage was to integrate the GCS-to-BigQuery loading operation.

The dependency was defined as:

```text
kaggle_to_gcs
       │
       │ dependency
       ▼
gcs_to_bigquery_raw
```

The `gcs_to_bigquery_raw` asset uses the output of the previous stage as its upstream dependency.

Conceptually:

```text
Kaggle
   │
   ▼
kaggle_to_gcs
   │
   ▼
GCS
   │
   ▼
gcs_to_bigquery_raw
   │
   ▼
BigQuery RAW
```

The dependency was added in `definitions.py` using:

```python
@asset(deps=[kaggle_to_gcs])
```

### Verify the asset graph

```bash
docker compose exec app bash -c \
"cd /module-2-project/dagster && \
dagster asset list -f definitions.py -d /module-2-project"
```

### Materialize the pipeline

The complete set of currently defined assets can be materialized using:

```bash
docker compose exec app bash -c \
"cd /module-2-project/dagster && \
dagster asset materialize -f definitions.py -d /module-2-project --select '*'"
```

This verifies that the upstream and downstream operations execute in the expected order.

---

# 5. Add BigQuery RAW → Profiling

The next pipeline operation was the profiling step.

The dependency was conceptually:

```text
gcs_to_bigquery_raw
          │
          ▼
   profile_orders
```

The profiling code analyses the BigQuery RAW orders table and generates a data-quality/profiling report.

The Dagster asset was defined in `definitions.py`.

### Verify the asset

```bash
docker compose exec app bash -c \
"cd /module-2-project/dagster && \
dagster asset list -f definitions.py -d /module-2-project"
```

### Materialize the profiling asset

```bash
docker compose exec app bash -c \
"cd /module-2-project/dagster && \
dagster asset materialize -f definitions.py -d /module-2-project --select profile_orders_asset"
```

The profiling operation is primarily used for **data-quality analysis** and does not produce a BigQuery table that should be consumed by the dbt transformation layer.

---

# 6. Integrate Dagster with dbt

The next step was to integrate the existing dbt project with Dagster.

First, verify that `dagster-dbt` is installed:

```bash
docker compose exec app python -c \
"import dagster_dbt; print('dagster-dbt is installed')"
```

If `dagster-dbt` is not installed, add it to `requirements.txt`.

Then rebuild the Docker image:

```bash
docker compose build app
```

Restart the container:

```bash
docker compose up -d --force-recreate
```

Verify the installation again:

```bash
docker compose exec app python -c \
"import dagster_dbt; print('dagster-dbt is installed')"
```

---

# 7. Connect Dagster to the dbt Assets

The dbt project is integrated into `definitions.py` using `dagster-dbt`.

The resulting logical flow is:

```text
gcs_to_bigquery_raw
        │
        ▼
profile_orders_asset

        │
        │   independent data-quality branch
        │
        └─────────────────────┐
                              │
                              ▼
                         dbt assets
                              │
                              ▼
                         stg_orders
                              │
                              ▼
                    monthly_order_volume
```

However, the profiling asset should **not** be made a dependency of the dbt assets merely because profiling occurs before or alongside dbt.

---

# 8. Why Profiling Is Not a dbt Dependency

The BigQuery RAW table is the actual data input for both profiling and dbt.

The structure is therefore better represented as:

```text
                    BigQuery RAW
                         │
                    olist_orders
                         │
                ┌────────┴────────┐
                │                 │
                ▼                 ▼
        profile_orders           dbt
             .py                  │
                │                 ▼
                │            stg_orders
                │                 │
                │                 ▼
                │        monthly_order_volume
                │
                ▼
          Data Quality
             Report
```

The profiling process does not create a data artifact that the dbt models require as input.

Therefore, making:

```text
profile_orders_asset
        ↓
       dbt
```

a dependency would introduce an artificial dependency into the data pipeline.

Instead, profiling and dbt are treated as **two downstream branches of the same BigQuery RAW data**.

---

# 9. Test BigQuery + dbt Flow

Before materializing the complete pipeline, the BigQuery and dbt portions can be tested independently.

The following command materializes the selected dbt asset and its required upstream dependencies:

```bash
docker compose exec app bash -c \
"cd /module-2-project/dagster && \
dagster asset materialize -f definitions.py \
-d /module-2-project \
--select '+monthly_order_volume'"
```

The expected logical flow is:

```text
BigQuery RAW
     │
     ▼
 dbt assets
     │
     ▼
stg_orders
     │
     ▼
monthly_order_volume
```

This provides a focused way to test the Dagster/dbt integration without executing the Kaggle extraction and BigQuery loading stages.

---

# 10. Link BigQuery RAW to dbt

At this stage, Dagster and dbt were successfully integrated.

The next requirement was to make the dependency explicit:

```text
gcs_to_bigquery_raw
          │
          ▼
       dbt assets
```

The dbt models consume the BigQuery RAW data, so Dagster needs to understand that the dbt assets should execute only after the RAW BigQuery load has completed.

The dbt-generated `AssetSpec` objects were therefore updated through the Dagster dbt translator/dependency configuration.

Conceptually:

```text
Kaggle
   │
   ▼
kaggle_to_gcs
   │
   ▼
GCS
   │
   ▼
gcs_to_bigquery_raw
   │
   ▼
dbt assets
   │
   ▼
stg_orders
   │
   ▼
monthly_order_volume
```

This ensures that Dagster understands the dependency between the external BigQuery RAW asset and the dbt assets.

---

# 11. Verify dbt Asset Dependencies

The dependency configuration can be inspected programmatically.

Run:

```bash
docker compose exec app python -c \
"import runpy; \
d=runpy.run_path('/module-2-project/dagster/definitions.py'); \
[(print('ASSET:', s.key, 'DEPS:', s.deps)) for s in d['olist_dbt_assets'].specs]"
```

This displays the generated dbt assets and their dependencies.

For example, the output can be inspected to confirm that the relevant dbt assets have the expected upstream dependency on the BigQuery RAW asset.

---

# 12. Verify the Complete Dagster Asset Graph

Once the dependencies have been configured, the complete Dagster asset list can be inspected:

```bash
docker compose exec app bash -c \
"cd /module-2-project/dagster && \
dagster asset list -f definitions.py -d /module-2-project"
```

The asset graph should represent the overall pipeline:

```text
Kaggle
   │
   ▼
kaggle_to_gcs
   │
   ▼
GCS
   │
   ▼
gcs_to_bigquery_raw
   │
   ├───────────────────┐
   │                   │
   ▼                   ▼
Profiling              dbt
   │                   │
   ▼                   ▼
Data Quality       stg_orders
Report                 │
                       ▼
              monthly_order_volume
                       │
                       ▼
                  Streamlit
```

---

# 13. Materialize the Dagster → dbt Flow

When only the existing BigQuery data and dbt transformations need to be tested, use:

```bash
docker compose exec app bash -c \
"cd /module-2-project/dagster && \
dagster asset materialize -f definitions.py \
-d /module-2-project \
--select '+monthly_order_volume'"
```

The `+` selection includes the upstream dependencies required by `monthly_order_volume`.

This is useful when testing the transformation portion without rerunning the complete ingestion pipeline.

---

# 14. Materialize the Complete Pipeline

To execute the complete Dagster flow:

```bash
docker compose exec app bash -c \
"cd /module-2-project/dagster && \
dagster asset materialize -f definitions.py \
-d /module-2-project \
--select '*'"
```

The complete execution path is:

```text
Kaggle
   │
   ▼
kaggle_to_gcs
   │
   ▼
GCS
   │
   ▼
gcs_to_bigquery_raw
   │
   ├──────────────────────┐
   │                      │
   ▼                      ▼
Profiling                 dbt
                          │
                          ▼
                     stg_orders
                          │
                          ▼
                 monthly_order_volume
                          │
                          ▼
                     Streamlit
```

Profiling and dbt are independent downstream branches from the BigQuery RAW layer.

---

# 15. Final Dagster Architecture

The final orchestration design is:

```text
                         ┌───────────────┐
                         │    Kaggle     │
                         └───────┬───────┘
                                 │
                                 ▼
                         ┌───────────────┐
                         │kaggle_to_gcs  │
                         └───────┬───────┘
                                 │
                                 ▼
                         ┌───────────────┐
                         │      GCS      │
                         └───────┬───────┘
                                 │
                                 ▼
                    ┌────────────────────────┐
                    │ gcs_to_bigquery_raw    │
                    └───────────┬────────────┘
                                │
                    ┌───────────┴───────────┐
                    │                       │
                    ▼                       ▼
              ┌───────────┐            ┌───────────┐
              │ Profiling │            │    dbt    │
              └─────┬─────┘            └─────┬─────┘
                    │                        │
                    ▼                        ▼
             Data Quality              stg_orders
                Report                     │
                                           ▼
                                  monthly_order_volume
                                           │
                                           ▼
                                      Streamlit
```

---

# 16. Key Orchestration Principle

The important design principle is that **Dagster manages dependencies between data-producing operations**, rather than simply forcing every operation to execute sequentially.

The pipeline therefore has two branches after the BigQuery RAW layer:

```text
                 BigQuery RAW
                      │
             ┌────────┴────────┐
             │                 │
             ▼                 ▼
         Profiling             dbt
             │                 │
             ▼                 ▼
      Data Quality Report   Analytics Models
```

Profiling provides data-quality information, while dbt produces the analytical data used by downstream consumers.

This allows both operations to be represented independently in Dagster while still sharing the same upstream RAW data.

---

# 17. Summary

Dagster was introduced incrementally rather than attempting to orchestrate the entire pipeline at once.

The implementation progressed through:

```text
1. Create Dagster framework
          ↓
2. Verify Dagster installation
          ↓
3. Add Kaggle → GCS asset
          ↓
4. Add GCS → BigQuery RAW asset
          ↓
5. Add profiling asset
          ↓
6. Install dagster-dbt
          ↓
7. Integrate dbt assets
          ↓
8. Connect BigQuery RAW → dbt
          ↓
9. Verify asset dependencies
          ↓
10. Materialize individual flows
          ↓
11. Materialize complete pipeline
```

The resulting architecture separates:

* **Ingestion** — Kaggle → GCS
* **Loading** — GCS → BigQuery RAW
* **Data Quality** — BigQuery RAW → Profiling
* **Transformation** — BigQuery RAW → dbt
* **Orchestration** — Dagster
* **Consumption** — Streamlit

Dagster therefore becomes the central orchestration layer that understands the relationships between the pipeline assets and can later be extended with scheduling, retries, monitoring, sensors, and automated execution.


## Pipeline Demo

### Terminal 1 — Start Docker and Dagster

```bash
# 1. Start Docker containers
docker compose up -d

# 2. Check container
docker ps

# 3. Verify Dagster
docker compose exec app dagster --version

# 4. Verify dbt
docker compose exec app dbt --version

# 5. Start Dagster UI
# 5. Start Dagster UI
docker compose exec app bash -c \
"cd /module-2-project/dagster && \
dagster dev -f definitions.py \
-d /module-2-project \
--host 0.0.0.0 \
--port 3000"
```

Keep Terminal 1 running.

Open Dagster UI in Safari:

```text
http://localhost:3000
```

---

### Terminal 2 — Start Streamlit

Open a second terminal:

```bash
# 6. Start Streamlit dashboard
docker compose exec app streamlit run /module-2-project/streamlit/app.py \
--server.address=0.0.0.0 \
--server.port=8501
```

Keep Terminal 2 running.

Open Streamlit dashboard in Safari:

```text
http://localhost:8501
```

---

### Terminal 3 — Execute the Pipeline

Open a third terminal.

```bash
# 7. Show all Dagster assets
docker compose exec app bash -c \
"cd /module-2-project/dagster && \
dagster asset list -f definitions.py -d /module-2-project"
```

```bash
# 8. Run the complete pipeline
docker compose exec app bash -c \
"cd /module-2-project/dagster && \
dagster asset materialize \
-f definitions.py \
-d /module-2-project \
--select '*'"
```

```bash
# 9. Verify dbt analytical output
docker compose exec app bash -c \
"cd /module-2-project/dagster && \
dagster asset materialize \
-f definitions.py \
-d /module-2-project \
--select '+monthly_order_volume'"
```

---

## Application URLs

```text
Dagster:    http://localhost:3000
Streamlit:  http://localhost:8501
```

## Demo Flow

```text
Kaggle Dataset
      ↓
   GCS Raw Data
      ↓
   BigQuery
      ↓
     dbt
      ↓
   Dagster
      ↓
Analytical Data
      ↓
   Streamlit
```

## Terminal Purpose

```text
Terminal 1 → Dagster UI server
Terminal 2 → Streamlit dashboard
Terminal 3 → Execute and verify pipeline assets
```

Keep Terminals 1 and 2 running while executing the pipeline commands from Terminal 3.


In [ ]:
# Preparing GCP VM

This section describes how to prepare a Google Cloud VM to run the **Olist Data Pipeline** using Docker Compose.

The VM is used as the execution environment for the project, replacing the local Mac development environment.

---

## 1. Create the GCP VM

Create a VM with the following configuration:

| Configuration    | Value               |
| ---------------- | ------------------- |
| Name             | `olist-dagster-vm`  |
| Region           | `asia-southeast1`   |
| Zone             | `asia-southeast1-a` |
| Machine type     | `E2 / e2-small`     |
| Operating System | Ubuntu LTS          |
| Boot disk        | ~20–30 GB           |

The VM will host the project repository and run the Docker Compose environment.

---

## 2. Install Docker and Docker Compose

Connect to the VM using SSH.

First check whether Docker is already installed:

```bash
docker --version
docker compose version
```

If Docker is not available, install Docker and Docker Compose:

```bash
sudo apt update

sudo apt install -y docker.io
sudo apt install -y docker-compose-v2
```

Verify the installation:

```bash
docker --version
docker compose version
```

---

## 3. Install Git

Git is required to clone the project repository from GitHub.

Install Git:

```bash
sudo apt update
sudo apt install -y git
```

Verify:

```bash
git --version
```

---

## 4. Clone the Project from GitHub

The project is transferred to the VM by cloning the GitHub repository.

### Architecture

```text
GitHub Repository
       │
       │ git clone
       ▼
GCP VM
       │
       └── module-2-project
              │
              └── Docker Compose
```

Create the project directory:

```bash
mkdir -p ~/projects/Module2_Bigdata_Proj
cd ~/projects/Module2_Bigdata_Proj
```

Clone the repository:

```bash
git clone <GITHUB_REPOSITORY_URL>
```

Enter the project directory:

```bash
cd module-2-project
```

Verify the project files:

```bash
ls -la
```

---

## 5. Configure GCP Credentials

The project requires access to Google Cloud resources such as:

* Google Cloud Project
* Google Cloud Storage (GCS)
* BigQuery

### Local Mac setup

On the local Mac, the Docker Compose configuration uses a service-account JSON key mounted into the container.

For example:

```text
Mac
 │
 └── Service Account JSON
          │
          ▼
      Docker Container
```

### VM setup

The VM can use its **own Google Cloud VM service account / Application Default Credentials** to communicate with GCP.

Therefore, the VM does **not** need the local Mac service-account JSON file.

The Docker Compose configuration should be adjusted so that it does not depend on:

```text
GCP_CREDENTIALS_PATH
```

or a locally stored service-account JSON file.

Instead, the container can use the VM's Google Cloud credentials.

```text
GCP VM
 │
 │ VM Service Account
 ▼
Google Cloud APIs
 │
 ├── GCS
 └── BigQuery
```

### Configure project and bucket information

Set the required environment variables for the project, for example:

```bash
export GCP_PROJECT_ID="my-project-bigdata-505312"
export GCS_BUCKET_NAME="olistcontainer"
```

Verify:

```bash
echo $GCP_PROJECT_ID
echo $GCS_BUCKET_NAME
```

If these values are defined through the project's `.env` file, ensure that the `.env` file contains the required configuration.

**Do not commit service-account JSON keys or other secrets to GitHub.**

---

## 6. Build the Docker Image

From the project directory:

```bash
cd ~/projects/Module2_Bigdata_Proj/module-2-project
```

Build the Docker image:

```bash
docker compose build
```

This builds the project image using the project's `Dockerfile` and Docker Compose configuration.

---

## 7. Verify the Docker Image

List all Docker images:

```bash
docker images
```

Then verify the images defined by Docker Compose:

```bash
docker compose images
```

You should see the project image listed.

---

## 8. Start the Docker Container

Start the project environment:

```bash
docker compose up -d
```

Verify that the container is running:

```bash
docker ps
```

Example:

```text
CONTAINER ID   IMAGE                  COMMAND               STATUS
xxxxxxxxxxxx   module-2-project-app   "tail -f /dev/null"   Up
```

The `tail -f /dev/null` command keeps the container running so that project components such as Python scripts, dbt, Dagster and Streamlit can be executed inside the container.

---

## 9. Verify the Project Environment

Enter the running container:

```bash
docker compose exec app bash
```

Check the project directory:

```bash
pwd
ls -la
```

Verify Python:

```bash
python --version
```

Verify dbt:

```bash
dbt --version
```

Verify the GCP environment variables:

```bash
echo $GCP_PROJECT_ID
echo $GCS_BUCKET_NAME
```

---

## VM Setup Summary

The complete setup flow is:

```text
                 GitHub
                    │
                    │ git clone
                    ▼
          ┌─────────────────────┐
          │      GCP VM         │
          │ olist-dagster-vm    │
          │                     │
          │ Ubuntu LTS           │
          │ Docker              │
          │ Docker Compose      │
          │ Git                 │
          └──────────┬──────────┘
                     │
                     ▼
             module-2-project
                     │
                     │ Docker Compose
                     ▼
          ┌─────────────────────┐
          │   Docker Container  │
          │                     │
          │ Python              │
          │ dbt                 │
          │ Dagster             │
          │ Streamlit           │
          └──────────┬──────────┘
                     │
             VM Service Account
                     │
          ┌──────────┴──────────┐
          ▼                     ▼
        GCS                  BigQuery
```

### Quick Setup Commands

For a new VM, the main commands are:

```bash
sudo apt update
sudo apt install -y git docker.io docker-compose-v2
```

Clone the project:

```bash
mkdir -p ~/projects/Module2_Bigdata_Proj
cd ~/projects/Module2_Bigdata_Proj

git clone <GITHUB_REPOSITORY_URL>

cd module-2-project
```

Build and start:

```bash
docker compose build
docker compose up -d
```

Verify:

```bash
docker images
docker compose images
docker ps
```

Enter the container:

```bash
docker compose exec app bash
```

Then verify:

```bash
python --version
dbt --version
```

---

## Important Notes

1. **Do not copy the Mac service-account JSON key to the VM.** Use the VM's Google Cloud identity where possible.

2. **Do not commit credentials to GitHub.** Keep secrets out of `.env`, Git history, and the repository unless they are non-sensitive configuration values.

3. The Docker image must be rebuilt after changes to the `Dockerfile` or dependencies:

```bash
docker compose build
```

4. After rebuilding, restart the environment if required:

```bash
docker compose up -d
```

5. Check the running container at any time with:

```bash
docker ps
```
